In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/CV Bootcamp'
OUT = DATA + '/outputs'
PRED = OUT + '/predictions'
import os, sys, json, numpy as np, pandas as pd, torch, torch.nn.functional as F
os.makedirs(PRED, exist_ok=True)
sys.path.append(DATA)
from model import ResNet18
assert torch.cuda.is_available(), 'Switch runtime to GPU'
dev = 'cuda'
mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
std = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1)
def load(f):
    return ((torch.tensor(np.load(f'{DATA}/data/{f}')).permute(0, 3, 1, 2).float().div(255) - mean) / std).to(dev)
X, XT = load('train_images.npy'), load('test_images.npy')
lab = pd.read_csv(f'{DATA}/data/labels.csv').set_index('index').sort_index()
tr = np.loadtxt(f'{DATA}/data/train_idx.txt', dtype=int)
va = np.loadtxt(f'{DATA}/data/val_idx.txt', dtype=int)
Xtr, Xva = X[torch.tensor(tr).to(dev)], X[torch.tensor(va).to(dev)]
del X
ytr = torch.tensor(lab.noisy_label.values[tr]).to(dev)
yva_n = torch.tensor(lab.noisy_label.values[va]).to(dev)
yva_c = torch.tensor(lab.clean_label.values[va]).to(dev)

def augment(x):
    n = x.size(0)
    x = F.pad(x, (4, 4, 4, 4))
    i, j = torch.randint(0, 9, (n,), device=dev), torch.randint(0, 9, (n,), device=dev)
    r = torch.arange(32, device=dev)
    x = x[torch.arange(n, device=dev)[:, None, None, None], torch.arange(3, device=dev)[None, :, None, None],
          (i[:, None] + r)[:, None, :, None], (j[:, None] + r)[:, None, None, :]]
    flip = torch.rand(n, device=dev) < 0.5
    x[flip] = x[flip].flip(3)
    return x

@torch.no_grad()
def predict(T):
    net.eval()
    with torch.autocast('cuda', dtype=torch.float16):
        return torch.cat([net(T[k:k + 1000]).float().argmax(1) for k in range(0, len(T), 1000)])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
torch.manual_seed(0); np.random.seed(0)
E, B, BETA, LAM = 60, 128, 0.7, 3.0
net = ResNet18(10).to(dev)
opt = torch.optim.SGD(net.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4, nesterov=True)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, E)
scaler = torch.amp.GradScaler('cuda')
tgt = torch.zeros(len(Xtr), 10, device=dev)
best, best_ep, best_pred = -1, 0, None
for ep in range(E):
    net.train()
    perm = torch.randperm(len(Xtr), device=dev)
    for s in range(0, len(perm), B):
        b = perm[s:s + B]
        with torch.autocast('cuda', dtype=torch.float16):
            out = net(augment(Xtr[b]))
        out = out.float()
        p = F.softmax(out, 1).clamp(1e-4, 1 - 1e-4)
        tgt[b] = BETA * tgt[b] + (1 - BETA) * (p / p.sum(1, keepdim=True)).detach()
        loss = F.cross_entropy(out, ytr[b]) + LAM * torch.log(1 - (tgt[b] * p).sum(1)).mean()
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    sched.step()
    pv = predict(Xva)
    vn, vc = (pv == yva_n).float().mean().item(), (pv == yva_c).float().mean().item()
    if vn > best:
        best, best_ep, best_pred = vn, ep + 1, predict(XT).cpu().numpy()
    print(ep + 1, f'val_noisy {vn:.4f}  val_clean {vc:.4f}')
pd.DataFrame({'id': np.arange(len(XT)), 'pred': best_pred}).to_csv(f'{PRED}/t4_elr.csv', index=False)
print(f'saved t4_elr.csv  best epoch {best_ep}  val_noisy {best:.4f}')

1 val_noisy 0.2976  val_clean 0.3654
2 val_noisy 0.3548  val_clean 0.4594
3 val_noisy 0.3924  val_clean 0.5268
4 val_noisy 0.3872  val_clean 0.5200
5 val_noisy 0.4120  val_clean 0.5696
6 val_noisy 0.4284  val_clean 0.5918
7 val_noisy 0.4330  val_clean 0.5828
8 val_noisy 0.4704  val_clean 0.6536
9 val_noisy 0.4534  val_clean 0.6138
10 val_noisy 0.4432  val_clean 0.6028
11 val_noisy 0.4876  val_clean 0.6730
12 val_noisy 0.4954  val_clean 0.6912
13 val_noisy 0.4686  val_clean 0.6510
14 val_noisy 0.5070  val_clean 0.7108
15 val_noisy 0.4982  val_clean 0.7110
16 val_noisy 0.5152  val_clean 0.7256
17 val_noisy 0.5054  val_clean 0.7320
18 val_noisy 0.5000  val_clean 0.7188
19 val_noisy 0.5112  val_clean 0.7154
20 val_noisy 0.5126  val_clean 0.7320
21 val_noisy 0.5104  val_clean 0.7504
22 val_noisy 0.5288  val_clean 0.7508
23 val_noisy 0.5262  val_clean 0.7490
24 val_noisy 0.5342  val_clean 0.7722
25 val_noisy 0.5280  val_clean 0.7482
26 val_noisy 0.5260  val_clean 0.7652
27 val_noisy 0.5344  

In [ ]:
print(json.dumps({'methods': [{'name': 'elr', 'file': 't4_elr.csv', 'val_acc_noisy_labels': round(best, 4)}],
                  'best_method': 'elr'}, indent=2))

{
  "methods": [
    {
      "name": "elr",
      "file": "t4_elr.csv",
      "val_acc_noisy_labels": 0.5794
    }
  ],
  "best_method": "elr"
}


In [ ]:
y = lab.noisy_label.values[tr]
files = ['t3A_train_probs.npy', 't3B_train_probs.npy', 'run1_train_probs.npy', 'run4_elr_train_probs.npy', 'run5_clean_train_probs.npy']
P = sum(np.load(f'{OUT}/{f}').astype(np.float32).mean(0) for f in files) / len(files)
pred, conf = P.argmax(1), P.max(1)
keep = pred == y
relabel = ~keep & (conf >= 0.5)
drop = ~keep & ~relabel
y_new = y.copy(); y_new[relabel] = pred[relabel]
ytr_new = torch.tensor(y_new).to(dev)
w = torch.tensor((~drop).astype(np.float32), device=dev)
print(f'keep {keep.sum()}  relabel {relabel.sum()}  drop {drop.sum()}')
clean_tr = lab.clean_label.values[tr]
print('label accuracy, report only:', (y == clean_tr).mean().round(4), '->', (y_new == clean_tr)[~drop].mean().round(4))

keep 28185  relabel 11786  drop 5029
label accuracy, report only: 0.5976 -> 0.8999


In [ ]:
torch.manual_seed(0); np.random.seed(0)
E, B = 60, 128
net = ResNet18(10).to(dev)
opt = torch.optim.SGD(net.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4, nesterov=True)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, E)
scaler = torch.amp.GradScaler('cuda')
best2, best2_ep, best2_pred = -1, 0, None
for ep in range(E):
    net.train()
    perm = torch.randperm(len(Xtr), device=dev)
    for s in range(0, len(perm), B):
        b = perm[s:s + B]
        with torch.autocast('cuda', dtype=torch.float16):
            out = net(augment(Xtr[b]))
        l = F.cross_entropy(out.float(), ytr_new[b], reduction='none')
        loss = (l * w[b]).sum() / w[b].sum().clamp(min=1e-6)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    sched.step()
    pv = predict(Xva)
    vn, vc = (pv == yva_n).float().mean().item(), (pv == yva_c).float().mean().item()
    if vn > best2:
        best2, best2_ep, best2_pred = vn, ep + 1, predict(XT).cpu().numpy()
    print(ep + 1, f'val_noisy {vn:.4f}  val_clean {vc:.4f}')
pd.DataFrame({'id': np.arange(len(XT)), 'pred': best2_pred}).to_csv(f'{PRED}/t4_relabel.csv', index=False)
print(f'saved t4_relabel.csv  best epoch {best2_ep}  val_noisy {best2:.4f}')

1 val_noisy 0.3180  val_clean 0.4160
2 val_noisy 0.4010  val_clean 0.5518
3 val_noisy 0.4262  val_clean 0.5946
4 val_noisy 0.4510  val_clean 0.6300
5 val_noisy 0.4622  val_clean 0.6670
6 val_noisy 0.4730  val_clean 0.6916
7 val_noisy 0.4998  val_clean 0.7094
8 val_noisy 0.5262  val_clean 0.7680
9 val_noisy 0.5318  val_clean 0.7496
10 val_noisy 0.5032  val_clean 0.7462
11 val_noisy 0.5092  val_clean 0.7322
12 val_noisy 0.5130  val_clean 0.7176
13 val_noisy 0.5032  val_clean 0.7076
14 val_noisy 0.5348  val_clean 0.7760
15 val_noisy 0.5256  val_clean 0.7608
16 val_noisy 0.5442  val_clean 0.7606
17 val_noisy 0.5416  val_clean 0.7808
18 val_noisy 0.5370  val_clean 0.7524
19 val_noisy 0.5362  val_clean 0.7738
20 val_noisy 0.5518  val_clean 0.7894
21 val_noisy 0.5360  val_clean 0.8006
22 val_noisy 0.5358  val_clean 0.7636
23 val_noisy 0.5512  val_clean 0.8148
24 val_noisy 0.5462  val_clean 0.7916
25 val_noisy 0.5510  val_clean 0.7994
26 val_noisy 0.5472  val_clean 0.7968
27 val_noisy 0.5392  

In [ ]:
m = [{'name': 'elr', 'file': 't4_elr.csv', 'val_acc_noisy_labels': 0.5794},
     {'name': 'relabel', 'file': 't4_relabel.csv', 'val_acc_noisy_labels': round(best2, 4)}]
print(json.dumps({'methods': m, 'best_method': max(m, key=lambda r: r['val_acc_noisy_labels'])['name']}, indent=2))

{
  "methods": [
    {
      "name": "elr",
      "file": "t4_elr.csv",
      "val_acc_noisy_labels": 0.5794
    },
    {
      "name": "relabel",
      "file": "t4_relabel.csv",
      "val_acc_noisy_labels": 0.5806
    }
  ],
  "best_method": "relabel"
}
